In [30]:
# 1. Install required libraries
!pip install -q huggingface_hub gradio pillow

import gradio as gr
from huggingface_hub import InferenceClient
from PIL import Image
from google.colab import userdata
import io

In [31]:
# 2. Setup Hugging Face Client
# Model: FLUX.1-schnell (optimized for speed and quality)
try:
    HF_TOKEN = userdata.get('HUGGINGFACEHUB_API_TOKEN')
    client = InferenceClient("black-forest-labs/FLUX.1-schnell", token=HF_TOKEN)
except Exception as e:
    print("Error: Please set your HF_TOKEN in Colab Secrets.")

In [32]:
# 3. Define the Generation Logic
def generate_image(prompt, seed):
    try:
        # Generate image using the serverless API
        image = client.text_to_image(
            prompt,
            seed=seed if seed != -1 else None,
            num_inference_steps=4 # 'schnell' models are optimized for 4 steps
        )
        return image
    except Exception as e:
        return None

In [33]:
import gradio as gr

# 4. Build Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# 🚀 High-Speed Image Gen Pipeline")
    gr.Markdown("Powered by **FLUX.1-schnell** via Hugging Face Serverless Inference.")

    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Prompt",
                placeholder="A cinematic shot of a robotic hummingbird made of glass...",
                lines=3
            )
            seed_slider = gr.Slider(
                minimum=-1, maximum=1000000, value=-1, step=1,
                label="Seed (-1 for random)"
            )
            generate_btn = gr.Button("Generate", variant="primary")

        with gr.Column():
            image_output = gr.Image(label="Result", type="pil")

    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_input, seed_slider],
        outputs=image_output
    )

In [ ]:
# 5. Launch (share=True creates a public link for 72 hours)
demo.launch(share=True, debug=True, theme=gr.themes.Ocean())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://92f2ef36ed4f9df6b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
